# Olist AI Analyst Copilot — SQL Analysis

## Day 1 + Day 2

Dataset: Olist Brazilian e-commerce dataset  
Database: DuckDB  
Purpose: SQL revision, business analysis, and portfolio documentation.

> **Important:** Monetary values are Brazilian reais (R$), not dollars.


In [1]:
import duckdb

con = duckdb.connect('../olist.duckdb')
con.sql('SHOW TABLES')


nameVARCHAR
raw_customers
raw_geolocation
raw_order_items
raw_order_payments
raw_order_reviews
raw_orders
raw_product_category_name_translation
raw_products
raw_sellers


# DAY 1 — Data exploration and core business metrics


## 1. Tables and row counts

**Business purpose:** Understand what data is available before analysis.


In [ ]:
con.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name;
""")


In [ ]:
con.sql("""
SELECT 'raw_orders' AS table_name, COUNT(*) AS row_count FROM raw_orders
UNION ALL SELECT 'raw_order_items', COUNT(*) FROM raw_order_items
UNION ALL SELECT 'raw_order_payments', COUNT(*) FROM raw_order_payments
UNION ALL SELECT 'raw_customers', COUNT(*) FROM raw_customers
UNION ALL SELECT 'raw_products', COUNT(*) FROM raw_products
UNION ALL SELECT 'raw_sellers', COUNT(*) FROM raw_sellers
UNION ALL SELECT 'raw_order_reviews', COUNT(*) FROM raw_order_reviews
UNION ALL SELECT 'raw_geolocation', COUNT(*) FROM raw_geolocation
UNION ALL SELECT 'raw_product_category_name_translation', COUNT(*) FROM raw_product_category_name_translation
ORDER BY table_name;
""")


## 2. Verify `raw_orders` grain

**Question:** Is there exactly one row per order?

**Required verification:** `COUNT(*) = COUNT(DISTINCT order_id)`.


In [ ]:
con.sql("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS distinct_orders
FROM raw_orders;
""")


**Verified:** 99,441 rows = 99,441 distinct orders, so `raw_orders` is exactly 1 row per order.


## 3. Order status breakdown

**Grain:** 1 row per order. Not every status represents a completed sale.


In [ ]:
con.sql("""
SELECT order_status, COUNT(*) AS orders,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_orders
FROM raw_orders
GROUP BY order_status
ORDER BY orders DESC;
""")


Known counts: delivered 96,478; shipped 1,107; canceled 625; unavailable 609; invoiced 314; processing 301; created 5; approved 2. Delivered share = **97.02%**.


## 4. Customer identity and repeat-customer rate


In [ ]:
con.sql("""
SELECT COUNT(DISTINCT customer_id) AS customer_ids,
       COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM raw_customers;
""")


In [ ]:
con.sql("""
WITH customer_orders AS (
    SELECT customer_unique_id, COUNT(*) AS order_count
    FROM raw_customers
    GROUP BY customer_unique_id
)
SELECT COUNT(*) FILTER (WHERE order_count > 1) AS repeat_customers,
       COUNT(*) AS unique_customers,
       ROUND(COUNT(*) FILTER (WHERE order_count > 1) * 100.0 / COUNT(*), 2) AS repeat_rate_pct
FROM customer_orders;
""")


**Known result:** 2,997 repeat customers / 96,096 unique customers = **3.12%**. This calculation uses all customer order records without an order-status filter; state that denominator/filter if used in final documentation.


## 5. Order date range


In [ ]:
con.sql("""
SELECT MIN(order_purchase_timestamp) AS first_order,
       MAX(order_purchase_timestamp) AS last_order
FROM raw_orders;
""")


Known range: **2016-09-04 21:15:19 to 2018-10-17 17:30:18**.


## 6. Monthly delivered orders


In [ ]:
con.sql("""
SELECT DATE_TRUNC('month', order_purchase_timestamp) AS month,
       COUNT(*) AS orders
FROM raw_orders
WHERE order_status = 'delivered'
GROUP BY month
ORDER BY month;
""")


Note: the raw dataset extends into October 2018, but the last month with delivered orders in this analysis is August 2018.


## 7. Category sales


In [ ]:
con.sql("""
SELECT p.product_category_name AS category,
       ROUND(SUM(i.price), 2) AS sales
FROM raw_orders o
JOIN raw_order_items i ON o.order_id = i.order_id
JOIN raw_products p ON i.product_id = p.product_id
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY sales DESC;
""")


Known finding: `health_beauty` is the top category by item sales at roughly **R$1.23M**. Item volume and sales value can tell different stories.


## 8. Sales by customer state


In [ ]:
con.sql("""
SELECT c.customer_state,
       ROUND(SUM(i.price), 2) AS sales
FROM raw_orders o
JOIN raw_customers c ON o.customer_id = c.customer_id
JOIN raw_order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY sales DESC;
""")


## 9. Average delivery time by state


In [ ]:
con.sql("""
SELECT c.customer_state,
       ROUND(AVG(DATE_DIFF('day', o.order_purchase_timestamp, o.order_delivered_customer_date)), 2) AS avg_delivery_days
FROM raw_orders o
JOIN raw_customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC;
""")


Known example: RR ≈ **29.34 days**, SP ≈ **8.7 days**, showing substantial geographic differences.


## 10. Late-delivery rate


In [ ]:
con.sql("""
WITH delivered AS (
    SELECT order_id, order_delivered_customer_date, order_estimated_delivery_date
    FROM raw_orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
)
SELECT COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) AS late_orders,
       COUNT(*) AS eligible_delivered_orders,
       ROUND(COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) * 100.0 / COUNT(*), 2) AS late_rate_pct
FROM delivered;
""")


Known result: **7,826 late / 96,470 eligible delivered orders = 8.11%**. Denominator = delivered orders with both actual and estimated delivery dates available.


## 11. Top 5 sellers by delivered item sales


In [ ]:
con.sql("""
SELECT i.seller_id, ROUND(SUM(i.price), 2) AS sales
FROM raw_orders o
JOIN raw_order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY i.seller_id
ORDER BY sales DESC
LIMIT 5;
""")


## 12. Correct payment total


In [ ]:
con.sql("""
SELECT ROUND(SUM(payment_value), 2) AS total_payment_value
FROM raw_order_payments;
""")


Known result: **R$16,008,872.12**. Calculate this directly from the payment table rather than joining it to item rows.


## 13. Double-counting demonstration


In [ ]:
con.sql("""
SELECT ROUND(SUM(p.payment_value), 2) AS inflated_payment_value
FROM raw_orders o
JOIN raw_order_items i ON o.order_id = i.order_id
JOIN raw_order_payments p ON o.order_id = p.order_id;
""")


Known bad-join result: **R$20,308,134.71**; inflation = **R$4,299,262.59**. An order with 3 items and 2 payments becomes 6 joined rows. Aggregate one side to the needed grain before joining; `DISTINCT` is not a generic fix.


# DAY 2 — Window functions and analytical SQL


## Window-function glossary

- `PARTITION BY` — splits rows into independent groups without collapsing them.
- `ROW_NUMBER()` — unique sequence within a partition.
- `RANK()` — ties share rank and later ranks can skip numbers.
- `LAG()` — previous available row.
- `LEAD()` — following row.
- `SUM() OVER` — windowed aggregate without collapsing rows.

**Important:** `LAG()` means previous available row, not automatically previous calendar month.


## 14. Top 3 categories by sales per state

**Grain:** state + category before ranking. **Filter:** delivered orders. **Metric:** `SUM(i.price)`.


In [ ]:
con.sql("""
WITH category_sales AS (
    SELECT c.customer_state, p.product_category_name, ROUND(SUM(i.price), 2) AS sales
    FROM raw_orders o
    JOIN raw_customers c ON o.customer_id = c.customer_id
    JOIN raw_order_items i ON o.order_id = i.order_id
    JOIN raw_products p ON i.product_id = p.product_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state, p.product_category_name
), ranked AS (
    SELECT customer_state, product_category_name AS category, sales,
           RANK() OVER (PARTITION BY customer_state ORDER BY sales DESC) AS category_rank
    FROM category_sales
)
SELECT customer_state, category, sales, category_rank
FROM ranked
WHERE category_rank <= 3
ORDER BY customer_state, category_rank;
""")


Expected observed result: 81 rows = 27 states × 3 ranks. Example SP: cama_mesa_banho R$472,238.07; beleza_saude R$453,916.48; relogios_presentes R$421,546.60. `RANK()` can return more than three rows on ties; use `ROW_NUMBER()` for exactly three rows per state.


## 15. Monthly GMV + MoM growth

Current notebook exercise: delivered orders; GMV = item price + freight value.

**Documentation warning:** the planned project glossary defines GMV as `SUM(price + freight_value)` for non-canceled orders. Keep the final project definition explicit and do not silently mix populations.


In [ ]:
con.sql("""
WITH monthly AS (
    SELECT DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
           ROUND(SUM(i.price + i.freight_value), 2) AS gmv
    FROM raw_orders o
    JOIN raw_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
), calendar AS (
    SELECT month FROM generate_series(
        (SELECT MIN(month) FROM monthly),
        (SELECT MAX(month) FROM monthly),
        INTERVAL '1 month'
    ) AS t(month)
), filled AS (
    SELECT c.month, COALESCE(m.gmv, 0) AS gmv
    FROM calendar c LEFT JOIN monthly m ON c.month = m.month
), monthly_lag AS (
    SELECT month, gmv, LAG(gmv) OVER (ORDER BY month) AS previous_gmv
    FROM filled
)
SELECT month, gmv, previous_gmv,
       ROUND((gmv - previous_gmv) / NULLIF(previous_gmv, 0) * 100, 2) AS mom_growth_pct
FROM monthly_lag
ORDER BY month;
""")


Why the calendar CTE matters: `generate_series` creates every month, the `LEFT JOIN` keeps empty months, and `COALESCE` turns missing GMV into zero. This makes MoM compare adjacent calendar months rather than adjacent rows that happen to exist.


## 16. First-to-second-order gap


In [ ]:
con.sql("""
WITH ranked_orders AS (
    SELECT c.customer_unique_id, o.order_purchase_timestamp,
           ROW_NUMBER() OVER (
               PARTITION BY c.customer_unique_id
               ORDER BY o.order_purchase_timestamp
           ) AS order_number
    FROM raw_orders o
    JOIN raw_customers c ON o.customer_id = c.customer_id
), date_orders AS (
    SELECT customer_unique_id,
           MAX(CASE WHEN order_number = 1 THEN order_purchase_timestamp END) AS first_order,
           MAX(CASE WHEN order_number = 2 THEN order_purchase_timestamp END) AS second_order
    FROM ranked_orders
    GROUP BY customer_unique_id
), order_gaps AS (
    SELECT customer_unique_id, first_order, second_order,
           DATE_DIFF('day', first_order, second_order) AS days_between
    FROM date_orders
)
SELECT AVG(days_between) AS avg_days_to_second_order
FROM order_gaps
WHERE second_order IS NOT NULL;
""")


Known result: **80.35 days**. One-order customers are excluded because they have no second-order date and therefore no valid gap. The explicit filter makes the population clear.


## 17. Monthly AOV


In [ ]:
con.sql("""
WITH order_values AS (
    SELECT o.order_id,
           DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
           SUM(i.price + i.freight_value) AS order_value
    FROM raw_orders o
    JOIN raw_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id, month
)
SELECT month, ROUND(AVG(order_value), 2) AS aov
FROM order_values
GROUP BY month
ORDER BY month;
""")


AOV must be calculated from one row per order. Averaging item rows would give orders with more items more weight.


# Key findings

1. Repeat customer rate: **3.12%**.
2. Late delivery rate: **8.11%** among 96,470 eligible delivered orders.
3. Average first-to-second-order gap: **80.35 days**.
4. Delivered orders: **96,478 / 99,441 = 97.02%**.
5. Direct payment total: **R$16,008,872.12**.
6. Bad item-payment join: **R$20,308,134.71**, inflating by **R$4,299,262.59**.
7. Delivery time differs substantially by state.
8. Sales value and item volume can produce different category rankings.


# Final SQL review checklist

- [ ] What is the grain of each source table?
- [ ] What is the expected join cardinality: 1:1, 1:M, or M:M?
- [ ] What is the output grain after each CTE?
- [ ] Is the order-status filter appropriate?
- [ ] Is the denominator explicitly defined?
- [ ] Could a one-to-many join duplicate monetary values?
- [ ] Are missing calendar periods handled for time-based growth?
- [ ] Are `NULL` and zero treated intentionally?
- [ ] Are monetary values labeled in R$?
- [ ] Have headline numbers been independently verified?


# Interview questions

1. Why can joining orders, items, and payments directly cause double-counting?
2. What is the difference between `ROW_NUMBER()` and `RANK()`?
3. Why use `customer_unique_id` instead of `customer_id` for repeat-customer analysis?
4. Why calculate AOV from one row per order?
5. Why can plain `LAG()` give the wrong MoM comparison when a month has no rows?
6. Why is the denominator important when reporting late-delivery rate?
7. How is `PARTITION BY` different from `GROUP BY`?
8. When would you use `ROW_NUMBER()` instead of `RANK()` for top-N?
